In [14]:
import pandas as pd
import copy
import os
import warnings
from datetime import datetime

pd.set_option('display.max_columns', None)
warnings.simplefilter("ignore", FutureWarning)

Data Extraction

In [15]:
folder = 'NCOA/'
files = os.listdir(folder)

ncoa_stage = pd.DataFrame(columns=['Account Number', 'NCOA Timeframe', 'Last Name', 'First Name', 'City', 'StateZip', 'Numeric Unknown', 'Street', 'FileName', 'ClientName'])

for file in files:
    if file.endswith('.txt') and file.__contains__('NcoaMovers'):
        file_path = os.path.join(folder, file)

        with open(file_path, 'r') as f:
            for line in f:
                row = line.strip().split('  ')

                data = [x for x in row if x != '']
                data += [file.replace('.txt', ''), file[file.index('_')+9:file.index('.')]]
                
                ncoa_stage.loc[len(ncoa_stage)] = data

In [16]:
for col in ncoa_stage.select_dtypes(include=['object']).columns:
    ncoa_stage[col] = ncoa_stage[col].str.strip()


In [17]:
ncoa_stage = ncoa_stage[['Account Number', 'NCOA Timeframe', 'Last Name', 
                         'First Name', 'Street', 'City', 'StateZip', 
                         'FileName', 'ClientName', 'Numeric Unknown']]

In [18]:
op_name = datetime.now().strftime("%Y-%m-%d").replace('-','')

ncoa_stage.to_csv(f"{op_name}_HCM_Hancock_Demo_Stage_Data.csv", index=False)
ncoa_stage.to_csv(f"{op_name}_HCM_Hancock_Demo_Stage_Data.txt", sep = '|', index=False)

Data Transformation

In [19]:
ncoa_core = copy.deepcopy(ncoa_stage)

In [20]:
ncoa_core['State'] = ncoa_core['StateZip'].str.extract(r'([A-Za-z]+)', expand=False)
ncoa_core['Zip'] = ncoa_core['StateZip'].str.extract(r'(\d+)', expand=False)

ncoa_core.drop('StateZip', axis=1, inplace=True)

In [21]:
ncoa_core['Zip'] = ncoa_core['Zip'].apply(lambda x: x[0:5] + '-' + x[5:] if len(x) > 5 else x)

In [22]:
ncoa_core['NCOA FileDate'] = ncoa_core['FileName'].apply(lambda x: pd.to_datetime(
                                                              x[x.index('_')+1:x.index('_')+5] + '-' +
                                                              x[x.index('_')+5:x.index('_')+7] + '-' +
                                                              x[x.index('_')+7:x.index('_')+9] + '-'))

In [23]:
ncoa_core = ncoa_core[['Account Number', 'NCOA Timeframe', 'Last Name', 
                         'First Name', 'Street', 'City', 'State', 'Zip', 
                         'FileName', 'NCOA FileDate', 'ClientName', 'Numeric Unknown']]

In [24]:
op_name = datetime.now().strftime("%Y-%m-%d").replace('-','')

ncoa_core.to_csv(f"{op_name}_HCM_Hancock_Demo_Core_Data.csv", index=False)
ncoa_core.to_csv(f"{op_name}_HCM_Hancock_Demo_Core_Data.txt", sep = '|', index=False)

Data Output

In [25]:
ncoa_op = ncoa_core[['Account Number', 'Last Name', 'First Name', 'Street', 'City', 'State', 'Zip', 'FileName', 'NCOA FileDate', 'ClientName']]

In [26]:
op_name = datetime.now().strftime("%Y-%m-%d").replace('-','')

ncoa_op.to_csv(f"{op_name}_HCM_Hancock_Demo_Update.csv", index=False)
ncoa_op.to_csv(f"{op_name}_HCM_Hancock_Demo_Update.txt", sep = '|', index=False)